# Mamba3Vision — Anemia Analysis (Kaggle)

**Simpler than DSA-Mamba3:** flat sequence of Mamba3 blocks (no VMamba hierarchy).  
Same two modes:
- `'task': 'classification'` → anemic / non-anemic  
- `'task': 'regression'`     → predict Hb value (g/dL)

**Run order:** Cell 1 → Cell 2 → Cell 3 → Cell 4 → Cell 5 → Cell 6 → Cell 7 → Cell 8 → Cell 9

In [ ]:
# ── Cell 1: Install dependencies & clone repo ──────────────────────────────
import subprocess, sys, os

# ── STEP 0: Triton version gate — run BEFORE importing torch ─────────────
# `tl.make_tensor_descriptor` was added in triton 3.1.0.
# triton 3.0.0 (shipped with PyTorch 2.4) does NOT have it.
# Strategy:
#   1. Check once.  If already OK → skip install (fast path).
#   2. If wrong: reinstall triton>=3.1.0, then restart the kernel.
#      On the second run the gate passes immediately.

def _triton_ok():
    try:
        import triton.language as _tl
        return hasattr(_tl, 'make_tensor_descriptor')
    except Exception:
        return False

if not _triton_ok():
    print('triton.language.make_tensor_descriptor missing — '
          'reinstalling triton >= 3.1.0 …')
    _r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         '--force-reinstall', '--no-deps', 'triton>=3.1.0'],
        capture_output=True, text=True,
    )
    if _r.returncode != 0:
        print('pip stderr:', _r.stderr.strip()[-800:])
    else:
        print('triton installed — restarting kernel so the new wheel is loaded …')

    # Restart the kernel; re-run this cell once the kernel is ready.
    try:
        import IPython
        IPython.Application.instance().kernel.do_shutdown(restart=True)
    except Exception:
        pass
    raise SystemExit(
        'Kernel restart requested.  Please re-run Cell 1 when the kernel is ready.'
    )

import triton as _triton
print(f'triton {_triton.__version__} — make_tensor_descriptor present ✓')

# ── Repo clone ────────────────────────────────────────────────────────────
REPO_URL = 'https://github.com/junaidmaqbool/mambathree.git'
REPO_DIR = './mambathree'

if not os.path.isdir(REPO_DIR):
    r = subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR],
                       capture_output=True, text=True)
    print(r.stdout or r.stderr)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=True)
    print('Repo already present — pulled latest.')

for pkg in ['einops', 'timm']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'causal-conv1d'], check=True)
except Exception:
    print('causal-conv1d skipped.')

# Fix torchvision circular import
import torch as _torch
_tv_map = {
    '2.0': '0.15.2', '2.1': '0.16.2', '2.2': '0.17.2', '2.3': '0.18.1',
    '2.4': '0.19.1', '2.5': '0.20.1', '2.6': '0.21.0',
}
_torch_maj_min = '.'.join(_torch.__version__.split('.')[:2])
_tv_pin = _tv_map.get(_torch_maj_min)
if _tv_pin:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    f'torchvision=={_tv_pin}'], check=True)
    print(f'torchvision pinned to {_tv_pin} for torch {_torch_maj_min}')
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    '--upgrade', 'torchvision'], check=True)
    print('torchvision upgraded (torch version unrecognized, using latest)')

# ── Patch repo so Mamba1/2 C extensions don't block the import ───────────
_init_path = os.path.join(REPO_DIR, 'mamba_ssm', '__init__.py')
_init_src = open(_init_path).read()
if 'selective_scan_fn = None' not in _init_src:
    print('Patching __init__.py…')
    with open(_init_path, 'w') as f:
        f.write('''\
__version__ = "2.3.1"

try:
    from mamba_ssm.ops.selective_scan_interface import selective_scan_fn, mamba_inner_fn
    from mamba_ssm.modules.mamba_simple import Mamba
    from mamba_ssm.modules.mamba2 import Mamba2
    from mamba_ssm.models.mixer_seq_simple import MambaLMHeadModel
except ImportError:
    selective_scan_fn = None
    mamba_inner_fn = None
    Mamba = None
    Mamba2 = None
    MambaLMHeadModel = None

try:
    from mamba_ssm.modules.mamba3 import Mamba3
except ImportError:
    Mamba3 = None
''')

_sscan_path = os.path.join(REPO_DIR, 'mamba_ssm', 'ops', 'selective_scan_interface.py')
_sscan_src = open(_sscan_path).read()
if 'selective_scan_cuda = None' not in _sscan_src:
    print('Patching selective_scan_interface.py…')
    _sscan_src = _sscan_src.replace(
        'import selective_scan_cuda',
        'try:\n    import selective_scan_cuda\nexcept ImportError:\n    selective_scan_cuda = None',
        1,
    )
    with open(_sscan_path, 'w') as f:
        f.write(_sscan_src)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('Setup complete.')


In [ ]:
# ── Cell 2: Configuration — edit everything here ──────────────────────────

CONFIG = {
    # ── Task ──────────────────────────────────────────────────────────────
    # 'classification' → anemic (1) / non-anemic (0)  — CrossEntropyLoss
    # 'regression'     → predict Hb value in g/dL     — HuberLoss
    'task': 'classification',

    # ── Dataset ───────────────────────────────────────────────────────────
    'dataset_dir': './imagehb/dataset/dataset/left_eye',
    'csv_path':    './merge_excel_1.csv',

    'image_col':    'Patient ID',
    'hb_col':       'Haemoglobin (gm/dL)',
    'label_col':    None,          # None → auto-derive from hb_threshold
    'hb_threshold': 11.5,          # hb < threshold → anemic (1)

    # ── Training ──────────────────────────────────────────────────────────
    'img_size':      224,
    'batch_size':    32,
    'num_epochs':    10,
    'learning_rate': 2e-4,
    'weight_decay':  0.05,
    'val_split':     0.2,
    'seed':          42,

    # ── Model (Mamba3Vision — flat, no hierarchy) ─────────────────────────
    # patch_size=16 → 14×14=196 tokens  (fast, low memory)
    # patch_size=8  → 28×28=784 tokens  (finer detail, more memory)
    'patch_size':      16,
    'dim':             256,   # single channel width throughout all blocks
    'depth':           6,     # number of Mamba3 blocks
    'd_state':         32,    # SSM state size; reduce to 16 if OOM
    'headdim':         64,    # must divide expand*dim=512 evenly
    'expand':          2,
    'chunk_size':      64,
    'rope_fraction':   0.5,
    'drop_path_rate':  0.1,

    'checkpoint_dir': '/kaggle/working/checkpoints_m3v',
    'results_dir':    '/kaggle/working',
}

print('Task       :', CONFIG['task'])
print('img_size   :', CONFIG['img_size'])
print('patch_size :', CONFIG['patch_size'],
      f"→ {(CONFIG['img_size']//CONFIG['patch_size'])**2} tokens")
print('dim        :', CONFIG['dim'], '  depth:', CONFIG['depth'])
print('batch      :', CONFIG['batch_size'], '  epochs:', CONFIG['num_epochs'])

In [ ]:
# ── Cell 3: Imports ────────────────────────────────────────────────────────
import os, sys, random, math, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix,
)

import matplotlib.pyplot as plt
import seaborn as sns

REPO_DIR = './mambathree'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Clear stale module cache before importing
for key in list(sys.modules.keys()):
    if key.startswith('mamba_ssm'):
        del sys.modules[key]

from mamba_ssm.models.mamba3_vision import Mamba3Vision

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CONFIG['seed'])

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    free, total = torch.cuda.mem_get_info()
    print(f'VRAM   : {free/1e9:.1f} GB free / {total/1e9:.1f} GB total')

In [ ]:
# ── Cell 4: Dataset & DataLoaders ──────────────────────────────────────────

_IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif'}


def _build_image_index(image_dir: str) -> dict:
    """Recursively scan image_dir and return {lowercase_stem: full_path}.

    Handles any extension case, subdirectories, and trailing _L / _R suffixes
    that may not appear in the CSV filename column.
    """
    index = {}
    for root, _, files in os.walk(image_dir):
        for fname in files:
            stem, ext = os.path.splitext(fname)
            if ext.lower() not in _IMG_EXTS:
                continue
            full = os.path.join(root, fname)
            index[stem.lower()] = full
            if '_' in stem:
                index.setdefault(stem.rsplit('_', 1)[0].lower(), full)
    return index


def _lookup(index: dict, filename: str):
    key = str(filename).lower()
    if key in index:
        return index[key]
    if '_' in key:
        stripped = key.rsplit('_', 1)[0]
        if stripped in index:
            return index[stripped]
    return None


def load_dataframe(cfg: dict) -> pd.DataFrame:
    """Load CSV, resolve image paths, drop any incomplete row."""
    df = pd.read_csv(cfg['csv_path'])

    missing_cols = [c for c in [cfg['image_col'], cfg['hb_col']] if c not in df.columns]
    if missing_cols:
        raise ValueError(
            f"Columns not found: {missing_cols}.\n"
            f"Available columns: {list(df.columns)}\n"
            "Update image_col / hb_col in CONFIG."
        )

    df = df.rename(columns={cfg['image_col']: 'filename', cfg['hb_col']: 'hb'})
    df['hb'] = pd.to_numeric(df['hb'], errors='coerce')

    if cfg['label_col'] is None or cfg['label_col'] not in df.columns:
        thr = cfg['hb_threshold']
        df['label'] = (df['hb'] < thr).astype('Int64')
        print(f'Label auto-derived: hb < {thr} → anemic (1)')
    else:
        df = df.rename(columns={cfg['label_col']: 'label'})

    n0 = len(df)
    df = df.dropna(subset=['filename', 'hb', 'label'])
    na_dropped = n0 - len(df)

    image_dir = cfg['dataset_dir']
    print(f'Scanning {image_dir} …')
    idx = _build_image_index(image_dir)
    print(f'  {len(idx):,} image stems found on disk.')

    df = df.copy()
    df['img_path'] = df['filename'].apply(lambda fn: _lookup(idx, fn))
    img_dropped = int(df['img_path'].isna().sum())
    df = df[df['img_path'].notna()].reset_index(drop=True)

    if na_dropped:
        print(f'Dropped {na_dropped:,} rows — NA filename / Hb / label')
    if img_dropped:
        print(f'Dropped {img_dropped:,} rows — image not found on disk')

    if len(df) == 0:
        avail = os.listdir(image_dir)[:10] if os.path.isdir(image_dir) else []
        raise RuntimeError(
            f"No valid samples remain!\n"
            f"  dataset_dir: {image_dir}\n"
            f"  First 10 files: {avail}\n"
            "Check dataset_dir and image_col in CONFIG."
        )

    print(f'Valid samples : {len(df):,}')
    if cfg['task'] == 'classification':
        counts = df['label'].value_counts().sort_index()
        print(f'  non-anemic (0): {counts.get(0, 0):,}')
        print(f'  anemic     (1): {counts.get(1, 0):,}')
    else:
        print(f'  Hb range : {df["hb"].min():.1f} – {df["hb"].max():.1f} g/dL'
              f'  (mean {df["hb"].mean():.1f})')
    return df


class AnemiaDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform, task: str = 'classification'):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self.task      = task

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        for offset in range(len(self.df)):
            row = self.df.iloc[(idx + offset) % len(self.df)]
            try:
                img = Image.open(row['img_path']).convert('RGB')
                if self.transform:
                    img = self.transform(img)
                return (img, int(row['label'])) if self.task == 'classification' \
                       else (img, float(row['hb']))
            except Exception:
                continue
        raise RuntimeError('No loadable image found in dataset')


def build_transforms(img_size: int):
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(img_size, scale=(0.75, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.15, hue=0.05),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    val_tf = transforms.Compose([
        transforms.Resize(int(img_size * 1.14)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    return train_tf, val_tf


# ── Build splits ───────────────────────────────────────────────────────────
df = load_dataframe(CONFIG)

stratify = df['label'] if CONFIG['task'] == 'classification' else None
train_df, val_df = train_test_split(
    df, test_size=CONFIG['val_split'],
    random_state=CONFIG['seed'], stratify=stratify,
)
print(f'Train: {len(train_df):,}  |  Val: {len(val_df):,}')

train_tf, val_tf = build_transforms(CONFIG['img_size'])

train_ds = AnemiaDataset(train_df, train_tf, CONFIG['task'])
val_ds   = AnemiaDataset(val_df,   val_tf,   CONFIG['task'])

train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'],
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG['batch_size'],
                          shuffle=False, num_workers=2, pin_memory=True)
print(f'Batches → train: {len(train_loader)}  val: {len(val_loader)}')

In [ ]:
# ── Cell 5: Model sanity check ─────────────────────────────────────────────

num_classes = 2 if CONFIG['task'] == 'classification' else 1

model = Mamba3Vision(
    img_size       = CONFIG['img_size'],
    patch_size     = CONFIG['patch_size'],
    in_chans       = 3,
    num_classes    = num_classes,
    dim            = CONFIG['dim'],
    depth          = CONFIG['depth'],
    d_state        = CONFIG['d_state'],
    headdim        = CONFIG['headdim'],
    expand         = CONFIG['expand'],
    chunk_size     = CONFIG['chunk_size'],
    rope_fraction  = CONFIG['rope_fraction'],
    drop_path_rate = CONFIG['drop_path_rate'],
).to(DEVICE)

print(f'Parameters : {model.num_parameters():,}')
print(f'Task       : {CONFIG["task"]}   num_classes={num_classes}')

with torch.no_grad():
    dummy = torch.randn(2, 3, CONFIG['img_size'], CONFIG['img_size']).to(DEVICE)
    out   = model(dummy)
    print(f'Forward OK  {tuple(dummy.shape)} → {tuple(out.shape)}')

del dummy, out, model
torch.cuda.empty_cache()

In [ ]:
# ── Cell 6: Training utilities ─────────────────────────────────────────────

def make_optimizer(model, lr: float, wd: float):
    no_decay = model.no_weight_decay()
    decay_params, nodecay_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if any(name.endswith(nd) for nd in no_decay) or 'pos_embed' in name:
            nodecay_params.append(p)
        else:
            decay_params.append(p)
    return torch.optim.AdamW(
        [{'params': decay_params,   'weight_decay': wd},
         {'params': nodecay_params, 'weight_decay': 0.0}],
        lr=lr,
    )


def train_one_epoch(model, loader, optimizer, loss_fn, device, task):
    model.train()
    total_loss = 0.0
    for images, targets in loader:
        images  = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs.squeeze(1), targets.float()) if task == 'regression' \
               else loss_fn(outputs, targets.long())
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * images.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, loss_fn, device, task):
    model.eval()
    total_loss, all_preds, all_targets = 0.0, [], []
    for images, targets in loader:
        images  = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        outputs = model(images)
        if task == 'regression':
            loss = loss_fn(outputs.squeeze(1), targets.float())
            all_preds.extend(outputs.squeeze(1).cpu().numpy())
        else:
            loss = loss_fn(outputs, targets.long())
            all_preds.extend(outputs.argmax(1).cpu().numpy())
        total_loss += loss.item() * images.size(0)
        all_targets.extend(targets.cpu().numpy())
    return total_loss / len(loader.dataset), np.array(all_preds), np.array(all_targets)


def compute_metrics(preds, targets, task):
    if task == 'classification':
        acc = accuracy_score(targets.astype(int), preds.astype(int)) * 100
        f1  = f1_score(targets.astype(int), preds.astype(int), average='binary') * 100
        return {'accuracy': acc, 'f1': f1}
    else:
        mae  = float(np.mean(np.abs(preds - targets)))
        rmse = float(np.sqrt(np.mean((preds - targets) ** 2)))
        ss_res = float(np.sum((targets - preds) ** 2))
        ss_tot = float(np.sum((targets - targets.mean()) ** 2))
        r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0
        return {'mae': mae, 'rmse': rmse, 'r2': r2}


print('Training utilities ready.')

In [ ]:
# ── Cell 7: Training loop ──────────────────────────────────────────────────

num_classes = 2 if CONFIG['task'] == 'classification' else 1
model = Mamba3Vision(
    img_size=CONFIG['img_size'],   patch_size=CONFIG['patch_size'],
    in_chans=3,                    num_classes=num_classes,
    dim=CONFIG['dim'],             depth=CONFIG['depth'],
    d_state=CONFIG['d_state'],     headdim=CONFIG['headdim'],
    expand=CONFIG['expand'],       chunk_size=CONFIG['chunk_size'],
    rope_fraction=CONFIG['rope_fraction'],
    drop_path_rate=CONFIG['drop_path_rate'],
).to(DEVICE)
print(f'Parameters : {model.num_parameters():,}')

os.makedirs(CONFIG['checkpoint_dir'], exist_ok=True)

optimizer = make_optimizer(model, CONFIG['learning_rate'], CONFIG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CONFIG['num_epochs'], eta_min=1e-6
)

if CONFIG['task'] == 'classification':
    loss_fn        = nn.CrossEntropyLoss()
    monitor_metric = 'f1'
    higher_better  = True
else:
    loss_fn        = nn.HuberLoss(delta=1.0)
    monitor_metric = 'mae'
    higher_better  = False

history    = {'train_loss': [], 'val_loss': [], 'val_metric': []}
best_val   = -float('inf') if higher_better else float('inf')
best_epoch = 0

hdr = f"{'Epoch':>5}  {'TrainLoss':>9}  {'ValLoss':>8}  {monitor_metric.upper():>10}  {'LR':>8}  Time"
sep = '─' * len(hdr)
print(hdr); print(sep)

for epoch in range(1, CONFIG['num_epochs'] + 1):
    t0 = time.time()

    train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, DEVICE, CONFIG['task'])
    val_loss, preds, targets_np = evaluate(model, val_loader, loss_fn, DEVICE, CONFIG['task'])
    metrics = compute_metrics(preds, targets_np, CONFIG['task'])
    scheduler.step()

    val_metric = metrics[monitor_metric]
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_metric'].append(val_metric)

    improved = (higher_better and val_metric > best_val) or \
               (not higher_better and val_metric < best_val)
    if improved:
        best_val   = val_metric
        best_epoch = epoch
        torch.save({
            'epoch': epoch, 'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'config': CONFIG, 'best_val': best_val,
            'monitor_metric': monitor_metric,
        }, os.path.join(CONFIG['checkpoint_dir'], 'best.pth'))
        star = ' *'
    else:
        star = ''

    lr_now  = scheduler.get_last_lr()[0]
    elapsed = time.time() - t0
    unit    = '%' if CONFIG['task'] == 'classification' else ''
    print(f"{epoch:>5}  {train_loss:>9.4f}  {val_loss:>8.4f}  "
          f"{val_metric:>9.3f}{unit}  {lr_now:>8.2e}  {elapsed:.0f}s{star}")

print(sep)
print(f'Best epoch: {best_epoch}  |  Best val {monitor_metric}: {best_val:.4f}')

In [ ]:
# ── Cell 8: Evaluate best checkpoint ──────────────────────────────────────
ckpt_path = os.path.join(CONFIG['checkpoint_dir'], 'best.pth')
ckpt      = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model_state'])
print(f"Loaded best checkpoint  epoch={ckpt['epoch']}  "
      f"{ckpt['monitor_metric']}={ckpt['best_val']:.4f}")

_, preds, targets_np = evaluate(model, val_loader, loss_fn, DEVICE, CONFIG['task'])
metrics = compute_metrics(preds, targets_np, CONFIG['task'])

print('\n── Validation metrics ──────────────────────')
for k, v in metrics.items():
    unit = '%' if k in ('accuracy', 'f1') else ''
    print(f'  {k:<12}: {v:.4f}{unit}')

if CONFIG['task'] == 'classification':
    print('\n── Classification report ───────────────────')
    print(classification_report(
        targets_np.astype(int), preds.astype(int),
        target_names=['non-anemic', 'anemic'],
    ))

In [ ]:
# ── Cell 9: Visualise results ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f"Mamba3Vision — {CONFIG['task'].capitalize()}", fontsize=14)

ax = axes[0]
ax.plot(history['train_loss'], label='train')
ax.plot(history['val_loss'],   label='val')
ax.axvline(best_epoch - 1, color='red', linestyle='--', alpha=0.6, label=f'best={best_epoch}')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Loss curves'); ax.legend()

ax = axes[1]
ax.plot(history['val_metric'])
ax.axvline(best_epoch - 1, color='red', linestyle='--', alpha=0.6)
unit = ' %' if CONFIG['task'] == 'classification' else ' g/dL'
ax.set_xlabel('Epoch')
ax.set_ylabel(f'{monitor_metric}{unit}')
ax.set_title(f'Val {monitor_metric}')

ax = axes[2]
if CONFIG['task'] == 'classification':
    cm = confusion_matrix(targets_np.astype(int), preds.astype(int))
    sns.heatmap(cm, annot=True, fmt='d', ax=ax,
                xticklabels=['non-anemic', 'anemic'],
                yticklabels=['non-anemic', 'anemic'],
                cmap='Blues')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title(f'Confusion matrix  (acc={metrics["accuracy"]:.1f}%)')
else:
    ax.scatter(targets_np, preds, alpha=0.35, s=10, color='steelblue')
    lo = min(float(targets_np.min()), float(preds.min())) - 0.5
    hi = max(float(targets_np.max()), float(preds.max())) + 0.5
    ax.plot([lo, hi], [lo, hi], 'r--', label='ideal')
    ax.set_xlabel('Actual Hb (g/dL)'); ax.set_ylabel('Predicted Hb (g/dL)')
    ax.set_title(f'Scatter  MAE={metrics["mae"]:.2f}  R²={metrics["r2"]:.3f}')
    ax.legend()

plt.tight_layout()
out_png = os.path.join(CONFIG['results_dir'], f'results_m3v_{CONFIG["task"]}.png')
plt.savefig(out_png, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out_png}')

## Quick reference

| Goal | Change in CONFIG |
|------|------------------|
| Switch to regression | `'task': 'regression'` then re-run cells 4–9 |
| More tokens (finer detail) | `'patch_size': 8` → 28×28 = 784 tokens |
| Wider / deeper model | `'dim': 384, 'depth': 8` |
| OOM / out-of-memory | Reduce `dim`, `depth`, or `batch_size` |
| Grayscale images | `in_chans=1` in Cell 5 & 7 |
| Custom Hb threshold | `'hb_threshold': 11.0` (WHO female) |

### Model file
```
mambathree/
  mamba_ssm/models/mamba3_vision.py   ← Mamba3Vision (this model)
  mamba_ssm/modules/mamba3.py         ← official Mamba3 SSM core
```

### Mamba3Vision vs DSA-Mamba3 (VSSM)
| | Mamba3Vision | DSA-Mamba3 |
|---|---|---|
| Architecture | Flat — one channel dim, N identical blocks | Hierarchical — 4 stages, channel doubling, patch merging |
| Spatial tokens | Fixed (196 for patch=16) | Shrinks per stage (56²→28²→14²→7²) |
| Parameters | ~7 M (dim=256, depth=6) | ~25 M (dims=[64,128,256,512]) |
| Speed | Faster | Slower |
| Best for | Quick experiments, small datasets | Larger datasets, publication accuracy |